# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 12.6 MB/s eta 0:00:00
dependencies ok


In [4]:
import re
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference,helper,numpy_helper,TensorProto

In [5]:
TASK_ID = "task115"
TASK_CANDIDATES = [
    Path(COMPETITION) / f"{TASK_ID}.json",
    Path.cwd() / f"{TASK_ID}.json",
    Path("/mnt/data") / f"{TASK_ID}.json",
]
TASK_PATH = next((p for p in TASK_CANDIDATES if p.exists()), None)
if TASK_PATH is None:
    raise FileNotFoundError(TASK_CANDIDATES)

OUT_DIR = Path.cwd() / f"{TASK_ID}_static_onnx_v2"
ONNX_PATH = OUT_DIR / f"{TASK_ID}.onnx"
SUMMARY_PATH = OUT_DIR / f"{TASK_ID}_validation_summary_v2.json"
SUBMISSION_PATH = Path.cwd() / "submission.zip"
OUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_SHAPE = [1, 10, 30, 30]
OUTPUT_SHAPE = [1, 10, 30, 30]
FORBIDDEN_OPS = {"Loop", "Scan", "NonZero", "Unique", "Script", "Function"}

with TASK_PATH.open() as f:
    task = json.load(f)

arc_gen_all = task.get("arc-gen", [])
arc_gen_compatible = [ex for ex in arc_gen_all if max(
    len(ex["input"]), len(ex["input"][0]), len(ex["output"]), len(ex["output"][0])
) <= 30]
excluded_arc_gen = len(arc_gen_all) - len(arc_gen_compatible)

indices = list(range(len(arc_gen_compatible)))
random.Random(20260710).shuffle(indices)
holdout_n = math.ceil(0.60 * len(indices))
holdout_indices = indices[:holdout_n]
development_indices = indices[holdout_n:]
arc_gen_holdout = [arc_gen_compatible[i] for i in holdout_indices]
arc_gen_development = [arc_gen_compatible[i] for i in development_indices]

print("task path:", TASK_PATH)
print({k: len(v) for k, v in task.items()})
print({
    "arc_gen_static_compatible": len(arc_gen_compatible),
    "arc_gen_static_excluded": excluded_arc_gen,
    "development_40pct": len(arc_gen_development),
    "holdout_60pct": len(arc_gen_holdout),
})

task path: /kaggle/input/competitions/neurogolf-2026/task115.json
{'train': 3, 'test': 1, 'arc-gen': 262}
{'arc_gen_static_compatible': 262, 'arc_gen_static_excluded': 0, 'development_40pct': 104, 'holdout_60pct': 158}


In [6]:

N = 30

def encode_grid(grid):
    arr = np.asarray(grid, dtype=np.int64)
    h, w = arr.shape
    assert h <= N and w <= N, (h, w)
    out = np.zeros((1, 10, N, N), dtype=np.float32)
    one_hot = np.eye(10, dtype=np.float32)[arr]
    out[0, :, :h, :w] = one_hot.transpose(2, 0, 1)
    return out

def expected_tensor(grid):
    return encode_grid(grid)

def tensor_contract(y, grid):
    h, w = len(grid), len(grid[0])
    outside = y.copy()
    outside[:, :, :h, :w] = 0
    outside_zero = bool(np.all(outside == 0))
    inside_sum = y[:, :, :h, :w].sum(axis=1)
    inside_one_hot = bool(np.all(inside_sum == 1.0))
    exact = bool(np.array_equal(y, expected_tensor(grid)))
    return exact, outside_zero, inside_one_hot

def fits_static_contract(ex):
    ih, iw = len(ex["input"]), len(ex["input"][0])
    oh, ow = len(ex["output"]), len(ex["output"][0])
    return max(ih, iw, oh, ow) <= N

def input_hash(grid):
    payload = json.dumps(grid, separators=(",", ":")).encode()
    return hashlib.sha256(payload).hexdigest()


In [7]:
class GridBase(nn.Module):
    def __init__(self):
        super().__init__()
        coord=torch.arange(N,dtype=torch.float32)
        self.register_buffer('coord',coord)
        self.register_buffer('coord_long',torch.arange(N,dtype=torch.long))
        rr=coord.view(N,1).expand(N,N)
        cc=coord.view(1,N).expand(N,N)
        self.register_buffer('rr',rr); self.register_buffer('cc',cc)
        self.register_buffer('colors',torch.arange(10,dtype=torch.long))
    def active_hw(self,x):
        active=x.sum(dim=1)>0
        H=active.any(dim=2).float().sum(dim=1)
        W=active.any(dim=1).float().sum(dim=1)
        return active,H,W
    def onehot_idx(self,idx):
        return (self.colors.view(1,10)==idx.view(-1,1)).float()
    def bounds(self,mask):
        rb=(mask>0).any(dim=2); cb=(mask>0).any(dim=1)
        cf=self.coord.view(1,N)
        rmin=torch.where(rb,cf,torch.full_like(cf,30.)).amin(dim=1)
        rmax=torch.where(rb,cf,torch.full_like(cf,-1.)).amax(dim=1)
        cmin=torch.where(cb,cf,torch.full_like(cf,30.)).amin(dim=1)
        cmax=torch.where(cb,cf,torch.full_like(cf,-1.)).amax(dim=1)
        return rmin,rmax,cmin,cmax
    def bounds_per_color(self,x):
        rowpres=(x>0).any(dim=3); colpres=(x>0).any(dim=2)
        cf=self.coord.view(1,1,N)
        rmin=torch.where(rowpres,cf,torch.full_like(cf,30.)).amin(dim=2)
        rmax=torch.where(rowpres,cf,torch.full_like(cf,-1.)).amax(dim=2)
        cmin=torch.where(colpres,cf,torch.full_like(cf,30.)).amin(dim=2)
        cmax=torch.where(colpres,cf,torch.full_like(cf,-1.)).amax(dim=2)
        return rmin,rmax,cmin,cmax
    def maps(self,start_r,start_c,h,w,reverse_c=False,reverse_r=False):
        b=start_r.shape[0]
        out=self.coord.view(1,N,1); inn=self.coord.view(1,1,N)
        sr=(start_r.view(b,1,1)+out) if not reverse_r else (start_r.view(b,1,1)-out)
        sc=(start_c.view(b,1,1)+out) if not reverse_c else (start_c.view(b,1,1)-out)
        rm=(inn==sr)&(out<h.view(b,1,1))
        cm=(inn==sc)&(out<w.view(b,1,1))
        return rm.float(),cm.float()
    def transform(self,x,rm,cm):
        t=torch.matmul(rm.unsqueeze(1),x)
        return torch.matmul(t,cm.transpose(1,2).unsqueeze(1))
    def canvas(self,h,w):
        return (self.rr.unsqueeze(0)<h[:,None,None])&(self.cc.unsqueeze(0)<w[:,None,None])
    def border_background(self,x):
        active,H,W=self.active_hw(x)
        border=active & ((self.rr[None]==0)|(self.cc[None]==0)|
                         (self.rr[None]==H[:,None,None]-1)|(self.cc[None]==W[:,None,None]-1))
        counts=(x*border[:,None].float()).sum(dim=(2,3))
        bg=counts.argmax(dim=1)
        return bg,self.onehot_idx(bg),active,H,W


class Task115v2(GridBase):
    def forward(self,x):
        b=x.shape[0]
        counts=x.sum(dim=(2,3)); present=counts>0
        # Directed color adjacency along each candidate ordering axis.
        # A_h[a,b] means color a occurs immediately left of color b;
        # A_v[a,b] means color a occurs immediately above color b.
        left=x[:,:,:,:-1].reshape(b,10,-1)
        right=x[:,:,:,1:].reshape(b,10,-1)
        top=x[:,:,:-1,:].reshape(b,10,-1)
        bottom=x[:,:,1:,:].reshape(b,10,-1)
        ah=torch.matmul(left,right.transpose(1,2))>0
        av=torch.matmul(top,bottom.transpose(1,2))>0
        offdiag=(self.colors.view(1,10,1)!=self.colors.view(1,1,10))
        ah=ah&offdiag&present[:,:,None]&present[:,None,:]
        av=av&offdiag&present[:,:,None]&present[:,None,:]

        # Along the true band-order axis, boundaries are predominantly one-way.
        # Jagged motion orthogonal to the order creates reciprocal color contacts.
        aht=ah.transpose(1,2); avt=av.transpose(1,2)
        h_one=(ah&(~aht)).float().sum(dim=(1,2))
        h_two=(ah&aht).float().sum(dim=(1,2))
        v_one=(av&(~avt)).float().sum(dim=(1,2))
        v_two=(av&avt).float().sum(dim=(1,2))
        hscore=h_one-3.0*h_two
        vscore=v_one-3.0*v_two
        horizontal_sequence=hscore>=vscore
        hs3=horizontal_sequence[:,None,None]
        adj=(hs3&ah)|((~hs3)&av)

        # Static transitive closure on the ten-color DAG. Four squaring steps
        # cover paths up to length sixteen, more than the maximum nine edges.
        reach=adj
        for _ in range(4):
            reach=reach|(torch.matmul(reach.float(),reach.float())>0)
            reach=reach&offdiag
        rank=reach.float().sum(dim=1).long()

        R=self.rr.view(1,1,N,N); C=self.cc.view(1,1,N,N)
        hs=horizontal_sequence.view(b,1,1,1)
        out=[]
        for color in range(10):
            hp=(R==0)&(C==rank[:,color].view(b,1,1,1))
            vp=(C==0)&(R==rank[:,color].view(b,1,1,1))
            pos=(hs&hp)|((~hs)&vp)
            out.append(pos.float()*present[:,color].view(b,1,1,1).float())
        return torch.cat(out,dim=1)


model = Task115v2().eval()
print(model)


Task115v2()


In [8]:
ADVERSARIAL_CASES = json.loads(r'''[{"input":[[3,3,3,3,3,3,3,3,3,3,3,3,3,3],[3,3,3,3,3,3,3,3,3,3,3,3,3,3],[3,3,3,3,3,3,3,3,3,3,3,3,3,3],[3,3,3,2,2,3,3,2,3,3,3,3,2,3],[3,3,2,2,2,2,2,2,3,2,3,3,2,2],[3,2,2,2,2,2,2,2,2,2,2,2,2,2],[2,2,2,2,2,2,2,2,2,2,2,2,2,2],[2,2,2,2,2,2,2,2,2,2,1,1,2,2],[2,1,1,2,2,1,1,1,2,2,1,1,1,2],[1,1,1,1,1,1,1,1,1,1,1,1,1,1],[1,1,1,1,1,1,1,1,1,1,1,8,1,1],[1,1,1,8,1,1,8,1,1,1,8,8,8,1],[8,8,8,8,8,8,8,8,1,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,8,8]],"output":[[3],[2],[1],[8]],"name":"C4 rotated layout"},{"input":[[6,6,6,6,2,2,2,9,9,9,9,4,4,4,4,4,7,7,7,7],[6,6,6,2,2,2,2,2,9,9,9,4,4,4,4,4,4,7,7,7],[6,6,6,2,2,2,2,9,9,9,9,9,4,4,4,4,7,7,7,7],[6,6,6,6,2,2,2,9,9,9,9,4,4,4,4,4,4,7,7,7],[6,6,6,2,2,2,2,9,9,9,9,4,4,4,4,4,7,7,7,7],[6,6,6,2,2,2,2,2,9,9,9,4,4,4,4,4,4,7,7,7],[6,6,6,6,2,2,2,9,9,9,9,4,4,4,4,4,7,7,7,7],[6,6,6,2,2,2,2,9,9,9,9,9,4,4,4,4,4,7,7,7],[6,6,6,2,2,2,2,9,9,9,9,4,4,4,4,4,7,7,7,7],[6,6,6,6,2,2,2,2,9,9,9,4,4,4,4,4,4,7,7,7],[6,6,6,2,2,2,2,9,9,9,9,4,4,4,4,4,7,7,7,7],[6,6,6,2,2,2,2,9,9,9,9,4,4,4,4,4,4,7,7,7],[6,6,6,6,2,2,2,9,9,9,9,9,4,4,4,4,7,7,7,7]],"output":[[6,2,9,4,7]],"name":"five vertical jagged bands"},{"input":[[6,6,6,6,6,6,6,6,6,6,6,6,6],[6,6,6,6,6,6,6,6,6,6,6,6,6],[6,6,6,6,6,6,6,6,6,6,6,6,6],[6,2,2,6,2,2,6,2,2,6,2,2,6],[2,2,2,2,2,2,2,2,2,2,2,2,2],[2,2,2,2,2,2,2,2,2,2,2,2,2],[2,2,2,2,2,2,2,2,2,2,2,2,2],[9,2,9,9,9,2,9,9,9,2,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9],[4,4,9,4,4,4,4,9,4,4,4,4,9],[4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4],[7,4,7,4,7,4,7,4,7,4,7,4,7],[7,7,7,7,7,7,7,7,7,7,7,7,7],[7,7,7,7,7,7,7,7,7,7,7,7,7],[7,7,7,7,7,7,7,7,7,7,7,7,7]],"output":[[6],[2],[9],[4],[7]],"name":"five horizontal jagged bands"}]''')

arc_hashes = {input_hash(ex["input"]) for ex in arc_gen_all}
assert all(input_hash(ex["input"]) not in arc_hashes for ex in ADVERSARIAL_CASES)
print("adversarial cases:", [ex.get("name", "unnamed") for ex in ADVERSARIAL_CASES])

def validate_torch(name, examples):
    ok = outside_ok = one_hot_ok = 0
    bad = []
    with torch.no_grad():
        for i, ex in enumerate(examples):
            y = model(torch.from_numpy(encode_grid(ex["input"]))).numpy()
            exact, outside, inside = tensor_contract(y, ex["output"])
            ok += int(exact)
            outside_ok += int(outside)
            one_hot_ok += int(inside)
            if not exact:
                bad.append(i)
    result = {
        "ok": ok,
        "total": len(examples),
        "bad_first10": bad[:10],
        "outside_zero_ok": outside_ok,
        "inside_one_hot_ok": one_hot_ok,
    }
    print(name, result)
    return result

torch_train = validate_torch("torch/train", task["train"])
torch_test = validate_torch("torch/test", task["test"])
torch_adversarial = validate_torch("torch/adversarial", ADVERSARIAL_CASES)
assert torch_train["ok"] == torch_train["total"]
assert torch_test["ok"] == torch_test["total"]
assert torch_adversarial["ok"] == torch_adversarial["total"]

adversarial cases: ['C4 rotated layout', 'five vertical jagged bands', 'five horizontal jagged bands']
torch/train {'ok': 3, 'total': 3, 'bad_first10': [], 'outside_zero_ok': 3, 'inside_one_hot_ok': 3}
torch/test {'ok': 1, 'total': 1, 'bad_first10': [], 'outside_zero_ok': 1, 'inside_one_hot_ok': 1}
torch/adversarial {'ok': 3, 'total': 3, 'bad_first10': [], 'outside_zero_ok': 3, 'inside_one_hot_ok': 3}


In [9]:
dummy = torch.from_numpy(encode_grid(task["train"][0]["input"]))

# Legacy TorchScript exporter is intentionally selected because it emits a compact,
# function-free static ONNX graph for these symbolic tensor programs.
torch.onnx.export(
    model,
    dummy,
    ONNX_PATH,
    input_names=["input"],
    output_names=["output"],
    opset_version=18,
    do_constant_folding=True,
    dynamic_axes=None,
    dynamo=False,
    external_data=False,
)

onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)
inferred = shape_inference.infer_shapes(onnx_model)

ops = Counter(node.op_type for node in onnx_model.graph.node)
forbidden = sorted(set(ops) & FORBIDDEN_OPS)
onnx_size = ONNX_PATH.stat().st_size
function_count = len(onnx_model.functions)

print("ONNX:", ONNX_PATH)
print("size bytes:", onnx_size)
print("ops:", dict(ops))
print("forbidden:", forbidden)
print("function_count:", function_count)

assert onnx_size < 1_400_000
assert not forbidden
assert function_count == 0

/tmp/ipykernel_16/2988166357.py:5: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX: /kaggle/working/task115_static_onnx_v2/task115.onnx
size bytes: 33764
ops: {'Constant': 67, 'ReduceSum': 6, 'Greater': 7, 'Slice': 4, 'Reshape': 25, 'Transpose': 4, 'MatMul': 6, 'And': 56, 'Unsqueeze': 4, 'Not': 4, 'Cast': 50, 'Mul': 12, 'Sub': 2, 'GreaterOrEqual': 1, 'Or': 15, 'Gather': 20, 'Equal': 20, 'Concat': 1}
forbidden: []
function_count: 0


In [10]:
session = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
input_shape = list(session.get_inputs()[0].shape)
output_shape = list(session.get_outputs()[0].shape)
print("input_shape:", input_shape)
print("output_shape:", output_shape)
assert input_shape == INPUT_SHAPE
assert output_shape == OUTPUT_SHAPE

def validate_onnx(name, examples):
    ok = outside_ok = one_hot_ok = 0
    bad = []
    for i, ex in enumerate(examples):
        y = session.run(None, {"input": encode_grid(ex["input"])})[0]
        exact, outside, inside = tensor_contract(y, ex["output"])
        ok += int(exact)
        outside_ok += int(outside)
        one_hot_ok += int(inside)
        if not exact:
            bad.append(i)
    result = {
        "ok": ok,
        "total": len(examples),
        "bad_first10": bad[:10],
        "outside_zero_ok": outside_ok,
        "inside_one_hot_ok": one_hot_ok,
    }
    print(name, result)
    return result

summary = {
    "task_id": TASK_ID,
    "revision": "v2-after-kaggle-feedback",
    "task_type": "topology-based global band-order extraction",
    "structural_rule": "Build directed color-contact graphs along both axes, select the axis with predominantly one-way boundaries, compute a static transitive closure, and emit colors in topological band order.",
    "improvements": ["replaces centroid sorting with structural directed adjacency", "static transitive closure supports all ten ARC colors", "orientation selected by one-way versus reciprocal boundary evidence", "works with color 0 as an actual band"],
    "model_family": "static symbolic feature tree",
    "input_shape": INPUT_SHAPE,
    "output_shape": OUTPUT_SHAPE,
    "onnx_size_bytes": onnx_size,
    "ops": dict(ops),
    "forbidden_ops": forbidden,
    "function_count": function_count,
    "arc_gen_total": len(arc_gen_all),
    "arc_gen_static_compatible": len(arc_gen_compatible),
    "arc_gen_static_excluded": excluded_arc_gen,
    "development_40pct_count": len(arc_gen_development),
    "holdout_60pct_count": len(arc_gen_holdout),
    "train": validate_onnx("onnx/train", task["train"]),
    "test": validate_onnx("onnx/test", task["test"]),
    "arc_gen_holdout_60pct": validate_onnx("onnx/arc-gen holdout", arc_gen_holdout),
    "adversarial": validate_onnx("onnx/adversarial", ADVERSARIAL_CASES),
}

for key in ["train", "test", "arc_gen_holdout_60pct", "adversarial"]:
    assert summary[key]["ok"] == summary[key]["total"], key
    assert summary[key]["outside_zero_ok"] == summary[key]["total"], key
    assert summary[key]["inside_one_hot_ok"] == summary[key]["total"], key

with SUMMARY_PATH.open("w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
print("summary:", SUMMARY_PATH)

input_shape: [1, 10, 30, 30]
output_shape: [1, 10, 30, 30]
onnx/train {'ok': 3, 'total': 3, 'bad_first10': [], 'outside_zero_ok': 3, 'inside_one_hot_ok': 3}
onnx/test {'ok': 1, 'total': 1, 'bad_first10': [], 'outside_zero_ok': 1, 'inside_one_hot_ok': 1}
onnx/arc-gen holdout {'ok': 158, 'total': 158, 'bad_first10': [], 'outside_zero_ok': 158, 'inside_one_hot_ok': 158}
onnx/adversarial {'ok': 3, 'total': 3, 'bad_first10': [], 'outside_zero_ok': 3, 'inside_one_hot_ok': 3}
{
  "task_id": "task115",
  "revision": "v2-after-kaggle-feedback",
  "task_type": "topology-based global band-order extraction",
  "structural_rule": "Build directed color-contact graphs along both axes, select the axis with predominantly one-way boundaries, compute a static transitive closure, and emit colors in topological band order.",
  "improvements": [
    "replaces centroid sorting with structural directed adjacency",
    "static transitive closure supports all ten ARC colors",
    "orientation selected by one-wa

In [11]:

if SUBMISSION_PATH.exists():
    SUBMISSION_PATH.unlink()

with zipfile.ZipFile(SUBMISSION_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(ONNX_PATH, arcname=f"{TASK_ID}.onnx")

print("submission:", SUBMISSION_PATH)
print("zip contents:")
with zipfile.ZipFile(SUBMISSION_PATH) as zf:
    infos = zf.infolist()
    for info in infos:
        print(info.filename, info.file_size)
    assert len(infos) == 1
    assert infos[0].filename == f"{TASK_ID}.onnx"


submission: /kaggle/working/submission.zip
zip contents:
task115.onnx 33764
